In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import diffrax as dfx
import arviz as az

from tb_macro.constants import AGE_STRATA, ISO3, START_TIME, END_TIME, LOCAL_OUTPUT_PATH
from tb_macro.epi import get_base_model, add_flows_to_model, initialise_pops
from tb_macro.inputs import load_demography, load_fertility, load_who_outcomes
from tb_macro.demography import prepare_pop_data_for_entries
from tb_macro.parameters import BASE_PARAMS
from tb_macro.plotting import plot_comp_distributions, plot_dynamic_mixing_matrix, plot_age_population_comparison

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [ ]:
# Model construction
group_popsize, death_rates, age_weights = load_demography(ISO3)
fert_padded = load_fertility(ISO3)
tsr, death_in_unsucc, who_mort = load_who_outcomes(ISO3)
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))
add_flows_to_model(
    epi_model, 
    disease_state,
    age_strat,
    clin_strat,
    infect_strat,
    age_weights,
    group_popsize,
    fert_padded,
    death_rates,
    tsr,
    death_in_unsucc,
    entry_times,
    entry_rates,
)
initialise_pops(epi_model, disease_state, age_strat, start_apops)

In [ ]:
solver_kwargs = {
    "max_steps": 4000,
    "stepsize_controller": dfx.PIDController(rtol=1e-5, atol=1e-5, dtmax=7.0),
    "adjoint": dfx.RecursiveCheckpointAdjoint(2048),
}

In [ ]:
run_id = "20260818T0648Z"
idata = az.from_netcdf(LOCAL_OUTPUT_PATH / f"{run_id}.nc")
sample = idata.posterior.stack(sample=("chain", "draw")).isel(sample=-1)
calib_params = {k: sample[k].values.item() for k in sample.data_vars}

In [ ]:
# Run for results
results = epi_model.run(BASE_PARAMS | calib_params, solver_kwargs=solver_kwargs)

In [ ]:
plot_comp_distributions(results, disease_state, age_strat, infect_strat, clin_strat, 1950.0, 2100.0, group_popsize)

In [ ]:
plot_age_population_comparison(results, group_popsize, age_strat, range(1970, 2021, 10))

In [ ]:
plot_dynamic_mixing_matrix(results["computed_values"]["dynamic_mm"], 1970.0, 10.0, 3)